# Stage 4: Exploratory Analysis & Business Insights
## Sportlytics Athletics — Professional Basketball Analytics

**Business Questions we answer:**
1. Which players are at highest risk of overload based on rolling workload?
2. Does player performance drop on back-to-back games?
3. How does travel distance interact with rest days to affect performance?
4. What training patterns correlate with optimal performance?

**Data Source:** Analytics outputs from Stage 2 batch pipeline (`/sportlytics/analytics/`)

In [1]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import warnings
warnings.filterwarnings('ignore')

# Reuse existing Spark session
spark = SparkSession.builder \
    .appName("Sportlytics-Exploration") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark session ready!")

# HDFS analytics path
ANALYTICS = "hdfs://namenode:9000/sportlytics/analytics"
print(f"Reading from: {ANALYTICS}")

Spark session ready!
Reading from: hdfs://namenode:9000/sportlytics/analytics


In [3]:
# Load all analytics Parquet files from Stage 2
rolling_workload = spark.read.parquet(f"{ANALYTICS}/rolling_workload.parquet")
back_to_back = spark.read.parquet(f"{ANALYTICS}/back_to_back_analysis.parquet")
travel_rest = spark.read.parquet(f"{ANALYTICS}/travel_rest_analysis.parquet")
training_load = spark.read.parquet(f"{ANALYTICS}/training_load_analysis.parquet")

# Quick summary
print("=== Datasets Loaded ===")
print(f"rolling_workload    : {rolling_workload.count():,} rows")
print(f"back_to_back        : {back_to_back.count():,} rows")
print(f"travel_rest         : {travel_rest.count():,} rows")
print(f"training_load       : {training_load.count():,} rows")

=== Datasets Loaded ===
rolling_workload    : 144,583 rows
back_to_back        : 900 rows
travel_rest         : 6 rows
training_load       : 1,800 rows


In [4]:
# Business Question 1: Top 10 highest workload players (injury risk)
print("=== Q1: Players at Highest Overload Risk ===\n")

top_workload = rolling_workload \
    .groupBy("player_id") \
    .agg(
        F.round(F.avg("rolling_5g_distance"), 1).alias("avg_rolling_distance_ft"),
        F.round(F.avg("rolling_5g_avg_hr"), 1).alias("avg_rolling_hr"),
        F.round(F.avg("rolling_5g_minutes"), 1).alias("avg_rolling_minutes"),
        F.count("game_id").alias("games_played")
    ) \
    .orderBy("avg_rolling_distance_ft", ascending=False) \
    .limit(10)

print("Top 10 Players by Rolling 5-Game Distance (Overload Risk):")
top_workload.show(truncate=False)

=== Q1: Players at Highest Overload Risk ===

Top 10 Players by Rolling 5-Game Distance (Overload Risk):
+---------+-----------------------+--------------+-------------------+------------+
|player_id|avg_rolling_distance_ft|avg_rolling_hr|avg_rolling_minutes|games_played|
+---------+-----------------------+--------------+-------------------+------------+
|PLR-0197 |2501.4                 |145.2         |70.2               |330         |
|PLR-0426 |2483.6                 |146.5         |66.4               |308         |
|PLR-0255 |2420.2                 |148.3         |64.8               |316         |
|PLR-0308 |2395.4                 |143.9         |66.9               |304         |
|PLR-0072 |2389.0                 |144.8         |63.3               |329         |
|PLR-0340 |2372.9                 |144.8         |64.3               |298         |
|PLR-0109 |2370.9                 |144.1         |63.1               |350         |
|PLR-0019 |2361.4                 |144.3         |62.9 

In [5]:
# Business Question 2: Back-to-back performance impact
print("=== Q2: Performance on Back-to-Back Games ===\n")

back_to_back.select(
    "back_to_back",
    "games_played",
    F.round("avg_plus_minus", 2).alias("avg_plus_minus"),
    F.round("avg_minutes", 1).alias("avg_minutes"),
    F.round("avg_distance_ft", 1).alias("avg_distance_ft"),
    F.round("avg_heart_rate", 1).alias("avg_heart_rate"),
    F.round("shooting_pct", 1).alias("shooting_pct")
).orderBy("back_to_back").show(truncate=False)

print("\nInsight: Compare rows where back_to_back=True vs False")
print("Lower plus_minus and shooting_pct on back-to-back nights = fatigue impact confirmed")

=== Q2: Performance on Back-to-Back Games ===

+------------+------------+--------------+-----------+---------------+--------------+------------+
|back_to_back|games_played|avg_plus_minus|avg_minutes|avg_distance_ft|avg_heart_rate|shooting_pct|
+------------+------------+--------------+-----------+---------------+--------------+------------+
|false       |459         |0.17          |6.9        |255.9          |147.2         |48.0        |
|false       |513         |-0.04         |7.0        |238.7          |144.1         |50.4        |
|false       |460         |-0.05         |7.0        |258.7          |146.8         |50.0        |
|false       |492         |0.4           |6.8        |244.8          |145.9         |51.4        |
|false       |461         |-0.62         |7.1        |260.9          |144.9         |49.6        |
|false       |475         |-0.03         |7.1        |258.4          |143.6         |39.5        |
|false       |524         |-0.58         |6.7        |256.3   

In [6]:
# Aggregate back-to-back vs normal games for clear comparison
print("=== Q2: Back-to-Back Impact Summary ===\n")

back_to_back.groupBy("back_to_back").agg(
    F.count("player_id").alias("total_records"),
    F.round(F.avg("avg_plus_minus"), 2).alias("avg_plus_minus"),
    F.round(F.avg("avg_minutes"), 1).alias("avg_minutes"),
    F.round(F.avg("avg_distance_ft"), 1).alias("avg_distance_ft"),
    F.round(F.avg("avg_heart_rate"), 1).alias("avg_heart_rate"),
    F.round(F.avg("shooting_pct"), 1).alias("shooting_pct")
).orderBy("back_to_back").show(truncate=False)

print("\nKey Finding:")
print("- If avg_plus_minus is LOWER on back_to_back=true → fatigue hurts performance")
print("- If shooting_pct is LOWER on back_to_back=true → fatigue hurts shooting")

=== Q2: Back-to-Back Impact Summary ===

+------------+-------------+--------------+-----------+---------------+--------------+------------+
|back_to_back|total_records|avg_plus_minus|avg_minutes|avg_distance_ft|avg_heart_rate|shooting_pct|
+------------+-------------+--------------+-----------+---------------+--------------+------------+
|false       |450          |0.0           |7.0        |249.4          |144.5         |50.0        |
|true        |450          |0.01          |7.0        |250.5          |144.6         |50.1        |
+------------+-------------+--------------+-----------+---------------+--------------+------------+


Key Finding:
- If avg_plus_minus is LOWER on back_to_back=true → fatigue hurts performance
- If shooting_pct is LOWER on back_to_back=true → fatigue hurts shooting


## Business Insight — Q2: Back-to-Back Games

**Finding:** Surprisingly, back-to-back games show minimal performance impact:
- Plus/minus: 0.00 (normal) vs 0.01 (back-to-back) — negligible difference
- Shooting %: 50.0% (normal) vs 50.1% (back-to-back) — no drop
- Distance: 249.4 ft (normal) vs 250.5 ft (back-to-back) — nearly identical

**Interpretation:** This suggests either:
1. Coaches are strategically resting high-workload players on back-to-back nights
2. The team has effective recovery protocols that minimize fatigue impact
3. Player rotation is managed well enough to maintain performance levels

**Recommendation:** Monitor individual high-workload players (from Q1) specifically 
on back-to-back nights rather than the team as a whole.

In [8]:
# Business Question 3: Travel distance vs rest days impact
print("Q3: Travel Distance and Rest Days Impact?\n")

travel_rest.select(
    "rest_days_since_last_game",
    "back_to_back",
    "total_records",
    F.round("avg_heart_rate", 1).alias("avg_heart_rate"),
    F.round("avg_distance_ft", 1).alias("avg_distance_ft"),
    F.round("avg_plus_minus", 2).alias("avg_plus_minus"),
    F.round("avg_travel_miles", 1).alias("avg_travel_miles"),
    F.round("shooting_pct", 1).alias("shooting_pct")
).orderBy("rest_days_since_last_game").show(truncate=False)

Q3: Travel Distance and Rest Days Impact?

+-------------------------+------------+-------------+--------------+---------------+--------------+----------------+------------+
|rest_days_since_last_game|back_to_back|total_records|avg_heart_rate|avg_distance_ft|avg_plus_minus|avg_travel_miles|shooting_pct|
+-------------------------+------------+-------------+--------------+---------------+--------------+----------------+------------+
|0                        |true        |37764        |144.6         |250.5          |0.0           |1490.9          |50.1        |
|1                        |false       |87979        |144.4         |249.6          |-0.01         |1379.8          |50.0        |
|2                        |false       |60260        |144.9         |249.1          |0.02          |1409.1          |50.0        |
|3                        |false       |39752        |144.6         |248.9          |-0.03         |1410.0          |49.6        |
|4                        |false       |

## Business Insight — Q3: Travel and Rest Days

**Finding:** Travel distance averages ~1,400 miles regardless of rest days,
but performance shows interesting patterns:

- **0 rest days (back-to-back):** avg_plus_minus = 0.0, shooting = 50.1%
- **1 rest day:** avg_plus_minus = -0.01, shooting = 50.0%
- **2 rest days:** avg_plus_minus = +0.02, shooting = 50.0%  
- **3 rest days:** avg_plus_minus = -0.03, shooting = 49.6% ← slight dip
- **4+ rest days:** avg_plus_minus = +0.09, shooting = 50.1% ← best performance

**Key Finding:** Players perform best with 4+ rest days regardless of travel distance.
The 3-day rest window shows a slight performance dip — possibly due to 
disrupted training rhythm.

**Recommendation:** Schedule high-travel road trips with at least 4 rest days 
before important games where possible.

In [9]:
# Business Question 4: Training load patterns
print("Q4: Training Session Patterns? \n")

training_load.groupBy("session_type").agg(
    F.round(F.avg("avg_intensity"), 2).alias("avg_intensity"),
    F.round(F.avg("avg_sleep_hours"), 1).alias("avg_sleep_hours"),
    F.round(F.avg("avg_duration_min"), 1).alias("avg_duration_min"),
    F.round(F.avg("avg_heart_rate"), 1).alias("avg_heart_rate"),
    F.sum("session_count").alias("total_sessions")
).orderBy("avg_intensity", ascending=False).show(truncate=False)

Q4: Training Session Patterns? 

+------------+-------------+---------------+----------------+--------------+--------------+
|session_type|avg_intensity|avg_sleep_hours|avg_duration_min|avg_heart_rate|total_sessions|
+------------+-------------+---------------+----------------+--------------+--------------+
|practice    |7.49         |7.2            |105.0           |127.7         |53684         |
|gym         |6.5          |7.2            |60.0            |111.3         |38484         |
|recovery    |2.5          |7.2            |39.9            |62.6          |38633         |
|film        |2.0          |7.2            |60.2            |60.2          |23260         |
+------------+-------------+---------------+----------------+--------------+--------------+



## Business Insight — Q4: Training Session Patterns

**Finding:** Four distinct training session types with clear intensity hierarchy:

| Session | Intensity | Duration | Heart Rate |
|---------|-----------|----------|------------|
| Practice | 7.49 (highest) | 105 min | 127.7 BPM |
| Gym | 6.50 | 60 min | 111.3 BPM |
| Recovery | 2.50 | 40 min | 62.6 BPM |
| Film | 2.00 (lowest) | 60 min | 60.2 BPM |

**Key Finding:** 
- Sleep averages exactly 7.2 hours across ALL session types — consistent recovery
- Practice sessions are 75% longer than gym sessions but nearly double the heart rate
- Recovery sessions are appropriately low intensity at 62.6 BPM

**Recommendation:** The 7.2 hour sleep average is below the recommended 8 hours 
for elite athletes. Improving sleep protocols could reduce injury risk for 
high-workload players identified in Q1.

In [11]:
# Stage 3 — Check Streaming Alerts Output
print("Stage 3: Streaming Alerts Check\n")

ANALYTICS = "hdfs://namenode:9000/sportlytics/analytics"

try:
    alerts = spark.read.parquet(f"{ANALYTICS}/streaming_alerts.parquet")
    print(f"Fatigue alerts written: {alerts.count():,} records")
    alerts.orderBy("avg_heart_rate", ascending=False).show(5, truncate=False)
except Exception as e:
    print(f"No streaming alerts yet — Stage 3 producer has not run.")
    print(f"Run 03-streaming-pipeline.ipynb and sportlytics_event_producer.py first.")
    print(f"This is expected if streaming has not been demonstrated yet.")

Stage 3: Streaming Alerts Check

No streaming alerts yet — Stage 3 producer has not run.
Run 03-streaming-pipeline.ipynb and sportlytics_event_producer.py first.
This is expected if streaming has not been demonstrated yet.


In [10]:
# Final Summary
print("=" * 55)
print("SPORTLYTICS BUSINESS INSIGHTS SUMMARY")
print("=" * 55)
print(f"Q1: Top overload risk player: PLR-0197")
print(f"    Avg 5-game distance: 2,501 ft | HR: 145.2 BPM")
print(f"Q2: Back-to-back impact: Minimal (well-managed rotation)")
print(f"Q3: Optimal rest: 4+ days yields best performance (+0.09)")
print(f"Q4: Sleep avg 7.2hrs — below 8hr elite recommendation")
print("=" * 55)
print("Pipeline: Stage1(HDFS) → Stage2(Spark) → Stage4(Insights)")

SPORTLYTICS BUSINESS INSIGHTS SUMMARY
Q1: Top overload risk player: PLR-0197
    Avg 5-game distance: 2,501 ft | HR: 145.2 BPM
Q2: Back-to-back impact: Minimal (well-managed rotation)
Q3: Optimal rest: 4+ days yields best performance (+0.09)
Q4: Sleep avg 7.2hrs — below 8hr elite recommendation
Pipeline: Stage1(HDFS) → Stage2(Spark) → Stage4(Insights)
